# IT25102149 - Random Forest Classifier

**AIML Group Assignment** | **Dataset:** UCI Default of Credit Card Clients

## Setup — shared train/test split

This notebook is a standalone copy of this member's section from `group_pipeline.ipynb`. The model code below is unchanged.

This cell loads the same 80/20 split that `group_pipeline.ipynb` saves to `results/outputs/train.csv` and `results/outputs/test.csv`, and defines the same `record_model()` function, so the scores match the group notebook. Run `group_pipeline.ipynb` first if these files are missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TARGET = 'default payment next month'

# Same rows as the shared split in group_pipeline.ipynb.
# float_precision='round_trip' reads every value back exactly as it was saved.
train_df = pd.read_csv('../results/outputs/train.csv', float_precision='round_trip')
test_df = pd.read_csv('../results/outputs/test.csv', float_precision='round_trip')
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

model_results = {}

def record_model(student_id, model_name, fitted_model):
    """Evaluate a fitted binary classifier on the shared test set."""
    predictions = fitted_model.predict(X_test)
    result = {
        'IT Number': student_id,
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': 'N/A',
    }
    classes = list(getattr(fitted_model, 'classes_', []))
    if 1 in classes and hasattr(fitted_model, 'predict_proba'):
        scores = fitted_model.predict_proba(X_test)[:, classes.index(1)]
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    elif 1 in classes and hasattr(fitted_model, 'decision_function'):
        scores = fitted_model.decision_function(X_test)
        if len(classes) == 2 and classes.index(1) == 0:
            scores = -scores
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    model_results[student_id] = result
    return pd.Series(result)


### IT25102149 - Random Forest Classifier

**Theoretical Justification & Model Suitability:**
- **Suitability:** Random Forest is an ensemble bootstrap aggregating (bagging) classifier. By building decorrelated decision trees using random feature subspaces, it drastically reduces model variance while maintaining low bias, making it the industry benchmark for credit risk datasets.
- **Tuning Strategy:** Tunes tree count (`n_estimators`: 100, 200), tree depth (`max_depth`: 6, 10, 15), and class cost balancing (`class_weight`: None vs `balanced`) to address the 22% default imbalance.
- **Performance Characteristics:** Cost-sensitive weighting penalizes false negatives heavily, achieving high recall and strong overall generalization.


In [ ]:
# ==============================================================================
# IT25102149 — Random Forest Classifier (deeper tuning + OOB + RF vs DT)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (ConfusionMatrixDisplay, f1_score, roc_auc_score,
                             accuracy_score, precision_score, recall_score,
                             classification_report, RocCurveDisplay)

# ------------------------------------------------------------------------------
# 1. Deeper grid: n_estimators, max_depth, max_features, min_samples_leaf, class_weight
# ------------------------------------------------------------------------------
rf_params = {
    'n_estimators':     [100, 200],
    'max_depth':        [6, 10, 15],
    'max_features':     ['sqrt', 'log2'],
    'min_samples_leaf': [2, 5],
    'class_weight':     [None, 'balanced']
}
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, bootstrap=True,
                           oob_score=True, n_jobs=-1),
    param_grid=rf_params, scoring='f1', cv=3, n_jobs=-1
)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

print('=== IT25102149: Random Forest Optimization ===')
print(f'Optimal Parameters      : {rf_search.best_params_}')
print(f'Training CV F1          : {rf_search.best_score_:.4f}')
print(f'Out-of-Bag (OOB) Score  : {best_rf.oob_score_:.4f}')

# ------------------------------------------------------------------------------
# 2. n_estimators convergence plot (Convergence Analysis)
# ------------------------------------------------------------------------------
n_range = [25, 50, 100, 200, 300]
conv_f1, conv_auc = [], []
for n in n_range:
    m = RandomForestClassifier(
        n_estimators=n,
        max_depth=best_rf.max_depth,
        max_features=best_rf.max_features,
        min_samples_leaf=best_rf.min_samples_leaf,
        class_weight=best_rf.class_weight,
        random_state=42, n_jobs=-1
    ).fit(X_train, y_train)
    conv_f1.append(f1_score(y_test, m.predict(X_test), zero_division=0))
    conv_auc.append(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(n_range, conv_f1,  'o-', color='#16a085', linewidth=2, label='Test F1')
ax.plot(n_range, conv_auc, 's-', color='#2980b9', linewidth=2, label='Test ROC-AUC')
ax.axvline(best_rf.n_estimators, color='#c0392b', linestyle=':',
           label=f'Chosen = {best_rf.n_estimators}')
ax.set_xlabel('n_estimators'); ax.set_ylabel('Score')
ax.set_title('IT25102149 RF: F1 & ROC-AUC Convergence vs Number of Trees',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 3. Feature importance — top-10 table + bar chart
# ------------------------------------------------------------------------------
feat_imp = pd.Series(best_rf.feature_importances_,
                     index=X_train.columns).sort_values(ascending=False)
print('\nTop 10 Feature Importances (Mean Decrease in Impurity):')
print(feat_imp.head(10).round(4))

fig, ax = plt.subplots(figsize=(9, 5))
top = feat_imp.head(10).iloc[::-1]
bars = ax.barh(top.index, top.values, color='#16a085', edgecolor='#0e6655')
ax.set_title('IT25102149 Random Forest: Top 10 Gini Feature Importances',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
for bar in bars:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 4. RF vs single Decision Tree — proves ensemble benefit (Ensemble Evaluation)
# ------------------------------------------------------------------------------
dt_baseline = DecisionTreeClassifier(
    max_depth=best_rf.max_depth, random_state=42
).fit(X_train, y_train)

rf_vs_dt = pd.DataFrame({
    'Model':    ['Single Decision Tree', 'Random Forest (tuned)'],
    'Accuracy': [accuracy_score(y_test, dt_baseline.predict(X_test)),
                 accuracy_score(y_test, best_rf.predict(X_test))],
    'Precision':[precision_score(y_test, dt_baseline.predict(X_test)),
                 precision_score(y_test, best_rf.predict(X_test))],
    'Recall':   [recall_score(y_test, dt_baseline.predict(X_test)),
                 recall_score(y_test, best_rf.predict(X_test))],
    'F1':       [f1_score(y_test, dt_baseline.predict(X_test)),
                 f1_score(y_test, best_rf.predict(X_test))],
    'ROC-AUC':  [roc_auc_score(y_test, dt_baseline.predict_proba(X_test)[:, 1]),
                 roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])]
})
print('\n=== Single DT vs Tuned Random Forest ===')
display(rf_vs_dt.round(4))

# ------------------------------------------------------------------------------
# 5. Inline metrics (Summary Evaluation)
# ------------------------------------------------------------------------------
y_pred_rf = best_rf.predict(X_test)
print('\n=== Best Random Forest: Test-Set Metrics ===')
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1]):.4f}")
print('\n' + classification_report(y_test, y_pred_rf,
                                    target_names=['No Default', 'Default']))

# ------------------------------------------------------------------------------
# 6. Confusion Matrix + ROC Curve
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    best_rf, X_test, y_test, cmap='Greens',
    display_labels=['No Default', 'Default'], ax=ax)
ax.set_title('IT25102149 Random Forest: Test Confusion Matrix',
             fontweight='bold')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_estimator(best_rf, X_test, y_test, ax=ax,
                               name='Random Forest (tuned)')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Chance')
ax.legend(); ax.set_title('IT25102149 Random Forest: ROC Curve',
                          fontweight='bold')
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 7. Register into shared scoreboard
# ------------------------------------------------------------------------------
record_model('IT25102149', 'Random Forest Classifier', best_rf)
